# Claude로 콘텐츠 모더레이션 필터 만들기
이 가이드에서는 Claude를 사용해 사용자 생성 텍스트를 위한 콘텐츠 모더레이션 필터를 만드는 방법을 알아봅니다. 핵심 아이디어는 모더레이션 규칙과 분류 범주를 프롬프트에 직접 정의해, 손쉽게 수정하고 실험할 수 있게 하는 것입니다.

## 기본 접근 방식
기본 접근 방식은 걸러 내고 싶은 범주(예: "ALLOW"와 "BLOCK")를 설명하고, 각 범주에 어떤 종류의 콘텐츠가 속하는지 상세한 설명이나 예시를 함께 담은 프롬프트를 Claude에 제공하는 것입니다. 그런 다음 분류할 사용자 생성 텍스트를 프롬프트의 일부로 삽입하고, 제시한 가이드라인에 따라 분류해 달라고 요청합니다.

프롬프트 구조 예시는 다음과 같습니다.

```text
You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- [Description or examples of content that should be blocked]

ALLOW CATEGORY:
- [Description or examples of content that is allowed]

Here is the user-generated text to categorize:
<user_text>{{USER_TEXT}}</user_text>

Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else.
```

실제로 사용할 때는 `{{USER_TEXT}}`를 분류할 실제 사용자 생성 텍스트로 바꾼 뒤, Claude API로 프롬프트를 전송하면 됩니다. Claude의 응답은 "ALLOW" 또는 "BLOCK" 중 하나여야 하며, 이는 여러분이 제시한 가이드라인에 따라 해당 텍스트를 어떻게 처리해야 하는지를 나타냅니다.

## 사용 예시
이 접근 방식을 실제로 사용하는 방법을 보여 주는 Python 예제 코드입니다:

In [ ]:
%pip install anthropic

In [2]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-haiku-4-5"


def moderate_text(user_text, guidelines):
    prompt_template = """
    You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

    {guidelines}

    Here is the user-generated text to categorize:
    <user_text>{user_text}</user_text>

    Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else.
    """

    # Format the prompt with the user text
    prompt = prompt_template.format(user_text=user_text, guidelines=guidelines)

    # Send the prompt to Claude and get the response
    response = (
        client.messages.create(
            model=MODEL_NAME, max_tokens=10, messages=[{"role": "user", "content": prompt}]
        )
        .content[0]
        .text
    )

    return response

다음은 이 함수를 사용해 사용자 댓글 배열을 모더레이션하는 예시입니다:

In [3]:
example_guidelines = """BLOCK CATEGORY:
    - Promoting violence, illegal activities, or hate speech
    - Explicit sexual content
    - Harmful misinformation or conspiracy theories

    ALLOW CATEGORY:
    - Most other content is allowed, as long as it is not explicitly disallowed
"""

user_comments = [
    "This movie was great, I really enjoyed it. The main actor really killed it!",
    "Delete this post now or you better hide. I am coming after you and your family.",
    "Stay away from the 5G cellphones!! They are using 5G to control you.",
    "Thanks for the helpful information!",
]

for comment in user_comments:
    classification = moderate_text(comment, example_guidelines)
    print(f"Comment: {comment}\nClassification: {classification}\n")

Comment: This movie was great, I really enjoyed it. The main actor really killed it!
Classification: ALLOW

Comment: Delete this post now or you better hide. I am coming after you and your family.
Classification: BLOCK

Comment: Stay away from the 5G cellphones!! They are using 5G to control you.
Classification: BLOCK

Comment: Thanks for the helpful information!
Classification: ALLOW



## 규칙 커스터마이징

이 접근 방식의 큰 장점 중 하나는 "BLOCK"과 "ALLOW" 범주에 대해 프롬프트에 적어 둔 설명이나 예시만 고치면 모더레이션 규칙을 손쉽게 바꿀 수 있다는 점입니다. 덕분에 필요와 취향에 맞게 필터링을 세밀하게 조정할 수 있습니다.

예를 들어 롤러코스터 애호가 포럼을 Claude에게 모더레이션하게 하면서 게시물이 주제에서 벗어나지 않도록 하고 싶다면, "ALLOW"와 "BLOCK" 범주 설명을 다음과 같이 바꾸면 됩니다:

In [4]:
rollercoaster_guidelines = """BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals
"""

post_titles = [
    "Top 10 Wildest Inversions on Steel Coasters",
    "My Review of the New RMC Raptor Coaster at Cedar Point",
    "Best Places to Buy Cheap Hiking Gear",
    "Rumor: Is Six Flags Planning a Giga Coaster for 2025?",
    "My Thoughts on the Latest Marvel Movie",
]

for title in post_titles:
    classification = moderate_text(title, rollercoaster_guidelines)
    print(f"Title: {title}\nClassification: {classification}\n")

Title: Top 10 Wildest Inversions on Steel Coasters
Classification: ALLOW

Title: My Review of the New RMC Raptor Coaster at Cedar Point
Classification: ALLOW

Title: Best Places to Buy Cheap Hiking Gear
Classification: BLOCK

Title: Rumor: Is Six Flags Planning a Giga Coaster for 2025?
Classification: ALLOW

Title: My Thoughts on the Latest Marvel Movie
Classification: BLOCK



## 생각의 사슬(CoT)로 성능 높이기

Claude의 콘텐츠 모더레이션 능력을 끌어올리는 기법 중 하나가 "생각의 사슬(chain-of-thought, CoT)" 프롬프팅입니다. 최종 출력만 내놓게 하는 대신, 추론 과정을 단계별 사고의 사슬로 풀어내도록 유도하는 방식입니다.

모더레이션에 생각의 사슬을 활용하려면, `<thinking>` 태그 안에서 판단 과정을 명확한 단계로 나누도록 프롬프트에 명시적으로 지시하면 됩니다. 예시는 다음과 같습니다:

In [8]:
cot_prompt = """You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals

First, inside of <thinking> tags, identify any potentially concerning aspects of the post based on the guidelines below and consider whether those aspects are serious enough to block the post or not. Finally, classify this text as either ALLOW or BLOCK inside <output> tags. Return nothing else.

Given those instructions, here is the post to categorize:

<user_post>{user_post}</user_post>"""

user_post = "Introducing my new band - Coaster Shredders. Check us out on YouTube!!"

response = (
    client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        messages=[{"role": "user", "content": cot_prompt.format(user_post=user_post)}],
    )
    .content[0]
    .text
)

print(response)

<thinking>
The post appears to be promoting a band rather than discussing rollercoasters, theme parks, or the amusement industry. This falls under the "spam, advertisements, or self-promotion" category, which is grounds for blocking the post.
</thinking>

<output>BLOCK</output>


## 예시를 추가해 성능 높이기
성능을 높이는 또 다른 기법은 프롬프트에 예시를 몇 개 추가하는 것입니다. 이렇게 하면 Claude에 초기 학습 데이터, 즉 "퓨샷 러닝(few-shot learning)"을 제공하는 셈이 되어 원하는 분류 기준을 더 잘 이해하게 됩니다. 글로 된 설명만으로는 범주의 경계가 분명하지 않은, 미묘하거나 애매한 사례에서 특히 유용합니다. 예시를 포함하도록 프롬프트 템플릿을 수정하는 방법은 다음과 같습니다:

In [9]:
examples_prompt = """You are a content moderation expert tasked with categorizing user-generated text based on the following guidelines:

BLOCK CATEGORY:
- Content that is not related to rollercoasters, theme parks, or the amusement industry
- Explicit violence, hate speech, or illegal activities
- Spam, advertisements, or self-promotion

ALLOW CATEGORY:
- Discussions about rollercoaster designs, ride experiences, and park reviews
- Sharing news, rumors, or updates about new rollercoaster projects
- Respectful debates about the best rollercoasters, parks, or ride manufacturers
- Some mild profanity or crude language, as long as it is not directed at individuals

Here are some examples:
<examples>
Text: I'm selling weight loss products, check my link to buy!
Category: BLOCK

Text: I hate my local park, the operations and customer service are terrible. I wish that place would just burn down.
Category: BLOCK

Text: Did anyone ride the new RMC raptor Trek Plummet 2 yet? I've heard it's insane!
Category: ALLOW

Text: Hercs > B&Ms. That's just facts, no cap! Arrow > Intamin for classic woodies too.
Category: ALLOW
</examples>

Given those examples, here is the user-generated text to categorize:
<user_text>{user_text}</user_text>

Based on the guidelines above, classify this text as either ALLOW or BLOCK. Return nothing else."""

user_post = "Why Boomerang Coasters Ain't It (Don't @ Me)"

response = (
    client.messages.create(
        model=MODEL_NAME,
        max_tokens=1000,
        messages=[{"role": "user", "content": examples_prompt.format(user_text=user_post)}],
    )
    .content[0]
    .text
)

print(response)

ALLOW
